In [1]:
import pandas as pd
import numpy as np

# ---
# ### Step 0: Setup and Configuration
# ---
print("--- Step 0: Initializing Setup ---")
RAW_DATA_FILE = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/notebooks/Skincare/cosmetics_sales_data.csv'
CLEANED_OUTPUT_FILE = '/Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/notebooks/Skincare/skincare_daily_sales.csv'
print(f"Input file: {RAW_DATA_FILE}")
print(f"Output file will be: {CLEANED_OUTPUT_FILE}\n")


# ---
# ### Step 1: Load Raw Data and Initial Inspection
# ---
print("--- Step 1: Loading Raw Data ---")
try:
    # Some CSV files have encoding issues; 'latin1' or 'ISO-8859-1' often works.
    df = pd.read_csv(RAW_DATA_FILE, encoding='latin1')
    print("Successfully loaded the dataset.")
except FileNotFoundError:
    print(f"ERROR: The file '{RAW_DATA_FILE}' was not found.")
    print("Please make sure the dataset is in the same directory as this script.")
    exit()

print("\n--- Initial Data Overview ---")
print("First 5 rows of the raw data:")
print(df.head())

print("\nDataset Information (dtypes and non-null counts):")
df.info()

print("\nInitial shape of the dataset (rows, columns):", df.shape)
print("-" * 50, "\n")


# ---
# ### Step 2: Data Cleaning and Basic Exploration
# ---
print("--- Step 2: Data Cleaning and Exploration ---")

# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
# We see many missing CustomerIDs and some missing Descriptions.
# We will drop rows with missing descriptions as they are unusable.
df.dropna(subset=['Description'], inplace=True)
print("\nDropped rows with missing 'Description'.")
print("New shape:", df.shape)

# Check for duplicate rows
print(f"\nNumber of duplicate rows found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print("Dropped duplicate rows.")
print("New shape:", df.shape)

# Explore the 'Country' column
print("\nTop 10 countries by number of transactions:")
print(df['Country'].value_counts().head(10))
print("NOTE: The data is heavily skewed towards the 'United Kingdom'.")
print("-" * 50, "\n")


# ---
# ### Step 3: Handle Returns (Negative Quantities)
# ---
# 

# [Image of a data cleaning process flowchart]

print("--- Step 3: Handling Returns ---")
print(f"Rows with negative 'Quantity' (returns): {len(df[df['Quantity'] <= 0])}")
print(f"Rows with positive 'Quantity' (sales): {len(df[df['Quantity'] > 0])}")

# For this analysis, we are only interested in sales, not returns.
df = df[df['Quantity'] > 0].copy()
print("\nRemoved all rows corresponding to returns.")
print("New shape:", df.shape)
print("-" * 50, "\n")


# ---
# ### Step 4: Filter to "Skincare" Category
# ---
print("--- Step 4: Filtering to Skincare Products ---")

# Define a list of keywords to identify skincare products.
# This list can be expanded for more fine-grained analysis.
skincare_keywords = [
    'cream', 'serum', 'lotion', 'moisturizer', 'cleanser', 'sunscreen',
    'mask', 'toner', 'exfoliat', 'balm', 'gel', 'facial', 'skin', 'eye'
]
# Create a regex pattern to find any of these words (case-insensitive)
regex_pattern = '|'.join(skincare_keywords)

print(f"Filtering 'Description' column for products containing: {skincare_keywords}")

# Ensure 'Description' is string and lowercase for consistent matching
df['Description'] = df['Description'].astype(str).str.lower()

# Create the filtered DataFrame
df_skincare = df[df['Description'].str.contains(regex_pattern, na=False)].copy()

print(f"\nTotal rows in dataset: {len(df)}")
print(f"Rows identified as skincare products: {len(df_skincare)}")
print(f"Retained {len(df_skincare) / len(df):.2%} of the sales data for our category.")
print("New shape of skincare-only data:", df_skincare.shape)

print("\nSample of identified skincare product descriptions:")
print(df_skincare['Description'].sample(5).to_list())
print("-" * 50, "\n")


# ---
# ### Step 5: Prepare Date Column for Time-Series Aggregation
# ---
print("--- Step 5: Preparing Date Column ---")
# Convert 'InvoiceDate' to a proper datetime object
df_skincare['InvoiceDate'] = pd.to_datetime(df_skincare['InvoiceDate'])

# We only need the date part for daily aggregation, so we extract it.
df_skincare['Date'] = df_skincare['InvoiceDate'].dt.date
print("Converted 'InvoiceDate' to datetime and extracted the 'Date'.")
print("Data types after conversion:")
print(df_skincare[['InvoiceDate', 'Date']].info())
print("-" * 50, "\n")


# ---
# ### Step 6: Aggregate Sales Data by Day
# ---
print("--- Step 6: Aggregating Sales by Day ---")

# Group by the new 'Date' column and sum the 'Quantity'
daily_sales = df_skincare.groupby('Date').agg(
    total_quantity_sold=('Quantity', 'sum')
).reset_index()

print("Created daily sales aggregation.")
print("Shape of daily sales data:", daily_sales.shape)
print("\nFirst 5 days of aggregated sales:")
print(daily_sales.head())

# To ensure our time series is continuous, we should check for missing dates.
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
date_range = pd.date_range(start=daily_sales['Date'].min(), end=daily_sales['Date'].max(), freq='D')
print(f"\nTime series spans from {daily_sales['Date'].min():%Y-%m-%d} to {daily_sales['Date'].max():%Y-%m-%d}.")
print(f"Total days in range: {len(date_range)}. Days with sales: {len(daily_sales)}.")

# Reindex the DataFrame to include all dates in the range, filling missing days with 0 sales.
daily_sales = daily_sales.set_index('Date').reindex(date_range).fillna(0).rename_axis('date').reset_index()

print("\nReindexed data to ensure a continuous daily timeline. Missing days filled with 0.")
print("New shape of daily sales data:", daily_sales.shape)
print("-" * 50, "\n")


# ---
# ### Step 7: Save the Processed Data
# ---
print("--- Step 7: Saving the Cleaned Data ---")
daily_sales.to_csv(CLEANED_OUTPUT_FILE, index=False)
print(f"Successfully saved the cleaned, aggregated data to '{CLEANED_OUTPUT_FILE}'.")

print("\n--- Final Data Sample ---")
print("First 5 rows of the final output file:")
print(daily_sales.head())
print("\nLast 5 rows of the final output file:")
print(daily_sales.tail())
print("\n--- PREPROCESSING COMPLETE ---")

--- Step 0: Initializing Setup ---
Input file: /Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/notebooks/Skincare/cosmetics_sales_data.csv
Output file will be: /Users/shashwatkushwaha/Desktop/AcadProj/Acad/MTP/priceengine2025/notebooks/Skincare/skincare_daily_sales.csv

--- Step 1: Loading Raw Data ---
Successfully loaded the dataset.

--- Initial Data Overview ---
First 5 rows of the raw data:
   Sales Person      Country                  Product        Date  Amount ($)  \
0   Lucas Verma       Canada            Aloe Vera Gel  2022-04-30     7897.13   
1   Ethan Reddy           UK            Aloe Vera Gel  2022-01-25    16376.88   
2  Ananya Gupta        India        Body Butter Cream  2022-08-22     5599.68   
3  Ananya Gupta  New Zealand  Salicylic Acid Cleanser  2022-08-26     2966.47   
4   Sophia Nair           UK        Body Butter Cream  2022-05-19     6828.68   

   Boxes Shipped  
0            358  
1            449  
2            264  
3            144  
4 

KeyError: ['Description']